## Note 4.1: Why 0.1+0.2=0.30000000000000004

### What you will learn

In this notebook, we:
- see why in float64 calculation 0.1+0.2 results in 0.30000000000000004
- while in other cases, 0.11+0.19 for example, results in 0.3

Let us first look at the fact: 

In [17]:
print("0.1  + 0.2  =", 0.1+0.2)
print("0.11 + 0.19 =", 0.11+0.19)
print("is 0.1  + 0.2  = 0.3 :", 0.1+0.2==0.3)
print("is 0.11 + 0.19 = 0.3 :", 0.11+0.19==0.3)

0.1  + 0.2  = 0.30000000000000004
0.11 + 0.19 = 0.3
is 0.1  + 0.2  = 0.3 : False
is 0.11 + 0.19 = 0.3 : True


## Explanation
Computers have only discretized set of numbers, represented by 64 bits in the case of float64.  Each discretized number has its bin, and any (continuous) real number within the bin is mapped to the discretized number, as in the following figure:  

We represent the discretized number as $\bar{r}$ to which a real number $r$ is mapped to.  

For example, a real number $0.3$ is mapped to the following discretized number $\overline{0.3}$: 

In [10]:
print(f"{0.3:.70f}")

0.2999999999999999888977697537484345957636833190917968750000000000000000


Let us next consider addition.  When we tell a computer to perform addition of two real numbers, $r_1+r_2$, the computer first maps $r_1$ and $r_2$ to their corresponding discretized numbers, $\bar{r}_1$ and $\bar{r}_2$, and then makes a sum, $\bar{r}_1 + \bar{r}_2$. The result is also mapped (rounded) to the nearby discretized number, and the final result is $\overline{\bar{r}_1 + \bar{r}_2}$.  

An important point is that it can happen that $\overline{\bar{r}_1 + \bar{r}_2} \neq \overline{r_1 + r_2}$.  Namely, when the rounding errors, $r_1-\bar{r}_1$ and $r_2-\bar{r}_2$, are unfortunately large, $\bar{r}_1 + \bar{r}_2$ can happen to reside in a bin different from the bin where $r_1+r_2$ resides.  

The example $0.1+0.2=0.30000000000000004$ is exactly the case: $\overline{0.1}+\overline{0.2}$ is rounded to a discretized number different from which 0.3 is mapped to (compare to the result of print(f"{0.3:.70f}") above). 

In [11]:
print(f"{0.1+0.2:.70f}")

0.3000000000000000444089209850062616169452667236328125000000000000000000


On the other hand, when the rounding errors are fortunately small, then $\overline{\bar{r}_1 + \bar{r}_2} = \overline{r_1 + r_2}$ holds.  The example $0.11 + 0.19 = 0.3$ is this case:

In [13]:
print(f"{0.11+0.19:.70f}")

0.2999999999999999888977697537484345957636833190917968750000000000000000


Compare the rounding errors for the operands in the two cases: 

In [14]:
print(f"{0.1:.70f}")
print(f"{0.11:.70f}")
print(f"{0.2:.70f}")
print(f"{0.19:.70f}")

0.1000000000000000055511151231257827021181583404541015625000000000000000
0.1100000000000000005551115123125782702118158340454101562500000000000000
0.2000000000000000111022302462515654042363166809082031250000000000000000
0.1900000000000000022204460492503130808472633361816406250000000000000000


We see that the rounding errors are all the same sign (negative), and they tend to add up in the calculation of addition.  But the rounding errors of $0.11$ and $0.19$ are smaller than $0.1$ and $0.2$, respectively, the addition $0.11+0.19$ happily equals to $0.3$ while $0.1+0.2$ unfortunately does not. 

Now we have grasped the essene.   The remaining part of this notebook is an appendix: it is a detailed and explicit demonstration of the bit calculation of the examples for interested readers.  

## Appendix: Looking through explicit bit manipulation

### Helper functions 
Below are functions to perform conversion between decimal representation and its float64 binary representation.

In [2]:
import numpy as np
from functools import partial

def decompose_float64(d: float) -> tuple[str, str, str]:  
    """
    Given a float64 number, this function returns its sign, exponent, fraction part as strings
    Args:
        d (float): Input floating-point number.
    Returns:
        tuple[str, str, str]:
            A tuple containing:
            - sign_str (str): The sign bit 
            - exponent_str (str): The 11-bit exponent
            - fraction_str (str): The 52-bit fraction
    """
    x = np.float64(d)
    x_int64 = x.view(np.int64)  # reinterpret the bits of x as those of int64 and returns the corresponding int64 number 
    binary_str = str(f"{x_int64:064b}")  # returns the string of binary representation of x_int64
    sign_str = binary_str[0]
    exponent_str = binary_str[1:12]
    fraction_str = binary_str[12:]
    return sign_str, exponent_str, fraction_str

def compose_float64(sign_str: str, exponent_str: str, fraction_str: str) -> float:
    """
    This is the "inverse" of decompose_float64
    """
    assert len(sign_str)==1 and len(exponent_str)==11 and len(fraction_str)==52
    assert set(sign_str)<= {'0', '1'} and set(exponent_str)<= {'0', '1'} and set(fraction_str)<= {'0', '1'}

    binary_str = sign_str + exponent_str + fraction_str
    x = int(binary_str, 2)   # interpret binary_str as binary representation of an integer and returns that integer
    x_int64 = np.int64(x)
    x_float64 = x_int64.view(np.float64)  # reinterpret the bits of x_int64 as those of float64 and returns the float64 number

    return x_float64.item()

### Bit manipulation
Now let us look at how bit manipulation is performed in the calculation of 0.1+0.2 and 0.11+0.19, respectively.

### Bit manipulation in the case of 0.1 + 0.2 = 0.30000000000000004

In [3]:
print(decompose_float64(0.1))
print(decompose_float64(0.2))

('0', '01111111011', '1001100110011001100110011001100110011001100110011010')
('0', '01111111100', '1001100110011001100110011001100110011001100110011010')


Name the former as A and the latter as B.  We connect the exponent and the fraction by '__':  
A = 01111111011____1001100110011001100110011001100110011001100110011010  
B = 01111111100____1001100110011001100110011001100110011001100110011010  
Prepend "1." to the fraction part, the result of which we call "1+f part":  
A = 01111111011__1.1001100110011001100110011001100110011001100110011010  
B = 01111111100__1.1001100110011001100110011001100110011001100110011010  
Make the exponent of A as the same as that of B, and shift the 1+f part to the right:  
A = 01111111100__0.11001100110011001100110011001100110011001100110011010  
B = 01111111100__1.1001100110011001100110011001100110011001100110011010  
Add the 1+f parts, and call the result C:   
C = 01111111100_10.01100110011001100110011001100110011001100110011001110   
Shift the 1+f part to the right, and increase the exponent part by 1.  
C = 01111111101_1.001100110011001100110011001100110011001100110011001110   
Round the fraction part:  
C = 01111111101_1.0011001100110011001100110011001100110011001100110100  
Delete "1.":  
C = 01111111101___0011001100110011001100110011001100110011001100110100

Now we can see the result is 0.30000000000000004

In [4]:
compose_float64("0", "01111111101", "0011001100110011001100110011001100110011001100110100")

0.30000000000000004

Remark:  
In the rounding process, "round to even" rule is applied.  In the intermediate result before the rounding,  
$$
C = 01111111101\_1.\underbrace{0011001100110011001100110011001100110011001100110011}_{\text{52 bits}}10,   
$$
the 53rd bit after the binary point is 1, and the bits after the 53rd bit are all 0.  In such a 'tie' case, the rounding is performed so that the 52nd bit after the binary point becomes 0.  

### Bit manipulation in the case of 0.11 + 0.19 = 0.3

In [5]:
print(decompose_float64(0.11))
print(decompose_float64(0.19))

('0', '01111111011', '1100001010001111010111000010100011110101110000101001')
('0', '01111111100', '1000010100011110101110000101000111101011100001010010')


Name the former as A and the latter as B. We connect the exponent and the fraction by '__':  
A = 01111111011___1100001010001111010111000010100011110101110000101001  
B = 01111111100___1000010100011110101110000101000111101011100001010010  
Prepend "1." to the fraction part, the result of which we call "1+f part":  
A = 01111111011_1.1100001010001111010111000010100011110101110000101001  
B = 01111111100_1.1000010100011110101110000101000111101011100001010010  
Make the exponent of A as the same as that of B, and shift the 1+f part to the right:  
A = 01111111100_0.11100001010001111010111000010100011110101110000101001  
B = 01111111100_1.1000010100011110101110000101000111101011100001010010  
Add the 1+f parts, and call the result C:  
C = 01111111100_10.01100110011001100110011001100110011001100110011001101  
Shift the 1+f part to the right, and increase the exponent part by 1:  
C = 01111111101_1.001100110011001100110011001100110011001100110011001101  
Round the fraction part:  
C = 01111111101_1.0011001100110011001100110011001100110011001100110011  
Delete "1.":  
C = 01111111101___0011001100110011001100110011001100110011001100110011  

Now we can see the result is 0.3

In [7]:
compose_float64('0','01111111101', '0011001100110011001100110011001100110011001100110011')

0.3

Remark:  
Now we have seen that 0.1 + 0.2 results in  
C = 01111111101___0011001100110011001100110011001100110011001100110100,  
which is 0.30000000000000004, while 0.11 + 0.19 results in   
C = 01111111101___0011001100110011001100110011001100110011001100110011,  
which is 0.3.  The fraction parts are different by 1 in the lowest bit.  